# Current Results Summary\n\nThis notebook summarizes the current status of the codebase and reproduces the main diagnostics from `notebooks/quickstart_debug.ipynb` using the **current implementation as-is**:\n\n1. Exact vs Gaussian backend agreement on small/intermediate sizes.\n2. Procedure A/B/C click-count statistics and observed discrepancies.\n3. A-vs-C Proposition-43 style statistical comparison in the `w>0` regime.\n4. Entanglement entropy along single trajectories + ensemble average (Gaussian backend).

In [ ]:
import numpy as np\nimport matplotlib.pyplot as plt\nfrom scipy.stats import chisquare\n\nfrom pps_qj import Simulator\nfrom pps_qj.types import SimulationConfig, Tolerances\nfrom pps_qj.models import single_projector_model, spin_chain_model, free_fermion_gaussian_model\nfrom pps_qj.observables import acceptance_fraction, entanglement_entropy_gamma\nfrom pps_qj.backends.gaussian import GaussianStateBackend\nfrom pps_qj.algorithms.waiting_time import RunContext, run_waiting_time_trajectory, run_pps_mc_trajectory\nfrom pps_qj.core.numerics import bracket_and_bisect\n\nsim = Simulator()\ntol = Tolerances()\n\ndef pmf_from_counts(arr, nmax=None):\n    arr = np.asarray(arr, dtype=int)\n    if arr.size == 0:\n        nmax = 0 if nmax is None else int(nmax)\n        out = np.zeros(nmax + 1)\n        out[0] = 1.0\n        return out\n    if nmax is None:\n        nmax = int(arr.max())\n    bc = np.bincount(arr, minlength=nmax + 1)\n    return bc / bc.sum()\n\ndef tv_distance(p, q):\n    n = max(len(p), len(q))\n    pp = np.pad(np.asarray(p, dtype=float), (0, n - len(p)))\n    qq = np.pad(np.asarray(q, dtype=float), (0, n - len(q)))\n    return 0.5 * np.sum(np.abs(pp - qq))

## 1) Exact vs Gaussian backend agreement (PPS-MC)

In [ ]:
backend_cfgs = [\n    dict(L=4, w=0.3, gamma=0.5, zeta=0.7, T=1.0, n=2000),\n    dict(L=4, w=0.5, gamma=0.5, zeta=0.6, T=1.5, n=2000),\n    dict(L=5, w=0.4, gamma=0.4, zeta=0.7, T=1.0, n=1200),\n]\n\nbackend_rows = []\nfor i, c in enumerate(backend_cfgs):\n    exact_model = spin_chain_model(L=c['L'], w=c['w'], gamma=c['gamma'])\n    gauss_model = free_fermion_gaussian_model(L=c['L'], w=c['w'], gamma=c['gamma'])\n\n    cfg_e = SimulationConfig(T=c['T'], zeta=c['zeta'], n_traj=c['n'], seed=1000+i, backend='exact', method='pps_mc', tolerances=tol)\n    cfg_g = SimulationConfig(T=c['T'], zeta=c['zeta'], n_traj=c['n'], seed=2000+i, backend='gaussian', method='pps_mc', tolerances=tol)\n\n    recs_e = sim.run_ensemble(cfg_e, exact_model)\n    recs_g = sim.run_ensemble(cfg_g, gauss_model)\n\n    clicks_e = np.array([r.n_clicks for r in recs_e])\n    clicks_g = np.array([r.n_clicks for r in recs_g])\n\n    nmax = max(clicks_e.max(), clicks_g.max())\n    p_e = pmf_from_counts(clicks_e, nmax=nmax)\n    p_g = pmf_from_counts(clicks_g, nmax=nmax)\n\n    mean_e = clicks_e.mean()\n    mean_g = clicks_g.mean()\n    se = np.sqrt(clicks_e.var(ddof=1)/len(clicks_e) + clicks_g.var(ddof=1)/len(clicks_g))\n\n    backend_rows.append(dict(\n        cfg=c,\n        mean_e=mean_e,\n        mean_g=mean_g,\n        four_se=4*se,\n        tv=tv_distance(p_e, p_g),\n        pass_4se=(abs(mean_e - mean_g) < 4*se),\n    ))\n\nprint('Exact vs Gaussian backend agreement (PPS-MC)')\nprint('-'*96)\nprint(f"{'L':>2} {'w':>4} {'g':>4} {'z':>4} {'T':>4} {'n':>5} {'mean_exact':>11} {'mean_gauss':>11} {'4*SE':>9} {'TV':>8} {'4SE?':>7}")\nfor r in backend_rows:\n    c = r['cfg']\n    print(f"{c['L']:2d} {c['w']:4.2f} {c['gamma']:4.2f} {c['zeta']:4.2f} {c['T']:4.1f} {c['n']:5d} {r['mean_e']:11.4f} {r['mean_g']:11.4f} {r['four_se']:9.4f} {r['tv']:8.4f} {str(r['pass_4se']):>7}")\nprint('-'*96)

## 2) Procedure A/B/C discrepancies (single-projector model)

In [ ]:
gamma_sp = 1.0\nzeta_sp = 0.5\nN_sp = 12000\nT_list = [0.5, 1.0, 2.0, 3.0]\n\npsi_q0 = np.array([1/np.sqrt(2), 1/np.sqrt(2)], dtype=np.complex128)\nmodel_sp = single_projector_model(gamma=gamma_sp, initial_state=psi_q0)\n\nabc_rows = []\nfor i, Ti in enumerate(T_list):\n    cfg_a = SimulationConfig(T=Ti, zeta=zeta_sp, n_traj=N_sp, seed=300+i, backend='exact', method='procedure_a', tolerances=tol)\n    cfg_b = SimulationConfig(T=Ti, zeta=zeta_sp, n_traj=N_sp, seed=400+i, backend='exact', method='procedure_b', tolerances=tol)\n    cfg_c = SimulationConfig(T=Ti, zeta=zeta_sp, n_traj=N_sp, seed=500+i, backend='exact', method='pps_mc', tolerances=tol)\n\n    ra = sim.run_ensemble(cfg_a, model_sp)\n    rb = sim.run_ensemble(cfg_b, model_sp)\n    rc = sim.run_ensemble(cfg_c, model_sp)\n\n    a_clicks = np.array([r.n_clicks for r in ra if r.accepted])\n    b_clicks = np.array([r.n_clicks for r in rb if r.accepted])\n    c_clicks = np.array([r.n_clicks for r in rc])\n\n    nmax = max(1, a_clicks.max(initial=0), b_clicks.max(initial=0), c_clicks.max(initial=0))\n    pa = pmf_from_counts(a_clicks, nmax=nmax)\n    pb = pmf_from_counts(b_clicks, nmax=nmax)\n    pc = pmf_from_counts(c_clicks, nmax=nmax)\n\n    abc_rows.append(dict(\n        T=Ti,\n        accA=acceptance_fraction(ra),\n        accB=acceptance_fraction(rb),\n        meanA=(a_clicks.mean() if a_clicks.size else np.nan),\n        meanB=(b_clicks.mean() if b_clicks.size else np.nan),\n        meanC=c_clicks.mean(),\n        tvCA=tv_distance(pc, pa),\n        tvCB=tv_distance(pc, pb),\n        pa=pa, pb=pb, pc=pc,\n    ))\n\nprint('Procedure A/B/C discrepancy summary')\nprint('-'*92)\nprint(f"{'T':>4} {'accA':>7} {'accB':>7} {'meanA':>8} {'meanB':>8} {'meanC':>8} {'TV(C,A)':>9} {'TV(C,B)':>9}")\nfor r in abc_rows:\n    print(f"{r['T']:4.1f} {r['accA']:7.3f} {r['accB']:7.3f} {r['meanA']:8.3f} {r['meanB']:8.3f} {r['meanC']:8.3f} {r['tvCA']:9.4f} {r['tvCB']:9.4f}")\nprint('-'*92)

In [ ]:
idx_show = 2  # T=2.0\nr = abc_rows[idx_show]\nn = np.arange(len(r['pc']))\nw = 0.28\n\nplt.figure(figsize=(9,4.5))\nplt.bar(n-w, r['pa'], width=w, alpha=0.8, label='Procedure A (accepted)')\nplt.bar(n,   r['pb'], width=w, alpha=0.8, label='Procedure B (accepted)')\nplt.bar(n+w, r['pc'], width=w, alpha=0.8, label='Procedure C (all)')\nplt.xlabel('click count n')\nplt.ylabel('probability')\nplt.title(f"A/B/C click-count PMFs at T={r['T']:.1f}, zeta={zeta_sp}")\nplt.legend()\nplt.tight_layout()\nplt.show()

## 3) A vs C statistics in the `w>0` regime (Gaussian backend, quickstart-style)

In [ ]:
L_t2, w_t2, gamma_t2 = 3, 0.5, 0.5\nT_t2, zeta_t2 = 2.0, 0.5\nN_born, N_pps = 30000, 12000\n\nmodel_t2 = free_fermion_gaussian_model(L=L_t2, w=w_t2, gamma=gamma_t2)\n\nrng_a = np.random.default_rng(42)\nn_A = []\nfor _ in range(N_born):\n    bg = GaussianStateBackend(model_t2, model_t2.initial_gamma.copy())\n    ctx = RunContext(backend=bg, rng=rng_a, T=T_t2, zeta=1.0, tol=tol)\n    rec = run_waiting_time_trajectory(ctx)\n    if rng_a.random() <= zeta_t2 ** rec.n_clicks:\n        n_A.append(rec.n_clicks)\nn_A = np.array(n_A, dtype=int)\n\nrng_c = np.random.default_rng(99)\nn_C = []\nfor _ in range(N_pps):\n    bg = GaussianStateBackend(model_t2, model_t2.initial_gamma.copy())\n    ctx = RunContext(backend=bg, rng=rng_c, T=T_t2, zeta=zeta_t2, tol=tol)\n    rec = run_pps_mc_trajectory(ctx)\n    n_C.append(rec.n_clicks)\nn_C = np.array(n_C, dtype=int)\n\nnmax = max(n_A.max(initial=0), n_C.max(initial=0), 6)\npA = pmf_from_counts(n_A, nmax=nmax)\npC = pmf_from_counts(n_C, nmax=nmax)\ntv_AC = tv_distance(pA, pC)\n\ncounts_A = np.bincount(n_A, minlength=nmax+1).astype(float)\ncounts_C = np.bincount(n_C, minlength=nmax+1).astype(float)\nmask = counts_A > 10\nchi2_p = np.nan\nif mask.sum() >= 2:\n    pref = counts_A[mask] / counts_A[mask].sum()\n    obs = counts_C[mask]\n    exp = pref * obs.sum()\n    _, chi2_p = chisquare(obs, f_exp=exp)\n\nprint('A vs C (Gaussian, w>0)')\nprint('-'*64)\nprint(f"N_A(accepted)={len(n_A)}, N_C={len(n_C)}")\nprint(f"meanA={n_A.mean():.4f}, meanC={n_C.mean():.4f}")\nprint(f"TV(A,C)={tv_AC:.4f}")\nprint(f"chi2 p-value (C vs A reference)={chi2_p:.4g}")\nprint('-'*64)

## 4) Entanglement entropy: single trajectories + ensemble average (Gaussian PPS-MC)

In [ ]:
def run_trajectory_with_ee(L, w, gam, T, zeta, l_sub, seed):\n    m = free_fermion_gaussian_model(L=L, w=w, gamma=gam)\n    bg = GaussianStateBackend(m, m.initial_gamma.copy())\n    bg.normalize()\n    rng = np.random.default_rng(seed)\n\n    t = 0.0\n    times_out = [0.0]\n    ee_out = [entanglement_entropy_gamma(bg.Gamma, l_sub)]\n\n    while t < T:\n        r = float(rng.random())\n        try:\n            tau = bracket_and_bisect(bg.survival, target=r, x0=0.0, x1=1.0, tol=tol)\n        except RuntimeError:\n            tau = float('inf')\n\n        if (not np.isfinite(tau)) or (t + tau >= T):\n            bg.propagate_no_click(max(T - t, 0.0))\n            t = T\n            times_out.append(t)\n            ee_out.append(entanglement_entropy_gamma(bg.Gamma, l_sub))\n            break\n\n        bg.propagate_no_click(tau)\n        t += tau\n\n        accepted = (zeta >= 1.0) or ((zeta > 0.0) and (float(rng.random()) <= zeta))\n        if accepted:\n            rates = bg.channel_rates()\n            total = rates.sum()\n            probs = rates/total if total > 0 else np.ones(len(rates))/len(rates)\n            cdf = np.cumsum(probs)\n            j = min(int(np.searchsorted(cdf, float(rng.random()), side='right')), len(rates)-1)\n            bg.apply_jump(j)\n\n        times_out.append(t)\n        ee_out.append(entanglement_entropy_gamma(bg.Gamma, l_sub))\n\n    return np.array(times_out), np.array(ee_out)\n\nL_ee, w_ee, gamma_ee = 12, 0.5, 0.5\nT_ee, zeta_ee = 8.0, 0.7\nl_sub = L_ee // 2\nN_ee = 40\n\nall_t, all_ee = [], []\nfor i in range(N_ee):\n    ts, ee = run_trajectory_with_ee(L_ee, w_ee, gamma_ee, T_ee, zeta_ee, l_sub, seed=7000+i)\n    all_t.append(ts)\n    all_ee.append(ee)\n\nt_grid = np.linspace(0, T_ee, 250)\nee_grid = np.zeros((N_ee, len(t_grid)))\nfor i in range(N_ee):\n    idx = np.searchsorted(all_t[i], t_grid, side='right') - 1\n    idx = np.clip(idx, 0, len(all_ee[i]) - 1)\n    ee_grid[i] = all_ee[i][idx]\n\nee_mean = ee_grid.mean(axis=0)\nee_std = ee_grid.std(axis=0)\n\nfig, ax = plt.subplots(1,2, figsize=(13,4.5))\nfor i in range(min(5, N_ee)):\n    ax[0].step(all_t[i], all_ee[i], where='post', alpha=0.8, lw=1.0)\nax[0].set_title('EE along single trajectories')\nax[0].set_xlabel('time')\nax[0].set_ylabel(f'S(ell={l_sub})')\n\nax[1].plot(t_grid, ee_mean, 'b-', lw=2, label='mean')\nax[1].fill_between(t_grid, ee_mean-ee_std, ee_mean+ee_std, color='b', alpha=0.2, label='±1 std')\nax[1].set_title(f'EE ensemble average (N={N_ee})')\nax[1].set_xlabel('time')\nax[1].set_ylabel(f'S(ell={l_sub})')\nax[1].legend()\nplt.tight_layout()\nplt.show()

## 5) Covariance-to-wavefunction conversion check (Exact vs Gaussian)

This section reconstructs many-body wavefunctions directly from Majorana covariance matrices for both backends (small `L`), then compares overlaps/fidelities.

- Build a common evolved state in both backends (same no-click time + same jump channel).
- Compute `Gamma_exact` from the exact many-body wavefunction and read `Gamma_gauss` from Gaussian backend.
- Reconstruct `psi_from_gamma` by minimizing covariance mismatch in Hilbert space.
- Compare reconstructed states between backends and against exact `psi`.

In [ ]:
from scipy.optimize import minimize
from pps_qj.models.spin_chain import jw_c_operators
from pps_qj.backends.exact import ExactStateBackend


def majorana_ops_jw(L):
    c_ops, cd_ops = jw_c_operators(L)
    gam = []
    for j in range(L):
        g0 = c_ops[j] + cd_ops[j]
        g1 = -1j * (c_ops[j] - cd_ops[j])
        gam.extend([g0, g1])
    return gam


def gamma_from_psi(psi, gammas):
    dim = len(gammas)
    G = np.zeros((dim, dim), dtype=float)
    rho = np.outer(psi, psi.conj())
    for a in range(dim):
        for b in range(dim):
            if a != b:
                G[a, b] = float(np.real(-1j * np.trace(rho @ gammas[a] @ gammas[b])))
    return G


def _vec_to_state(x, dim):
    psi = x[:dim] + 1j * x[dim:]
    nrm = np.linalg.norm(psi)
    if nrm < 1e-14:
        psi = np.zeros(dim, dtype=np.complex128)
        psi[0] = 1.0
        return psi
    psi = psi / nrm
    k0 = int(np.argmax(np.abs(psi)))
    phase = np.angle(psi[k0])
    psi = psi * np.exp(-1j * phase)
    return psi


def reconstruct_state_from_gamma(target_gamma, L, n_starts=10, seed=0):
    rng = np.random.default_rng(seed)
    dim = 2 ** L
    gammas = majorana_ops_jw(L)

    def loss(x):
        psi = _vec_to_state(x, dim)
        G = gamma_from_psi(psi, gammas)
        return float(np.sum((G - target_gamma) ** 2))

    best = None
    best_loss = np.inf
    for k in range(n_starts):
        x0 = rng.normal(size=2 * dim)
        if k == 0:
            x0[:] = 0.0
            x0[0] = 1.0
        res = minimize(loss, x0, method='L-BFGS-B', options={'maxiter': 700})
        if res.fun < best_loss:
            best_loss = float(res.fun)
            best = res.x.copy()

    psi_best = _vec_to_state(best, dim)
    G_best = gamma_from_psi(psi_best, gammas)
    return psi_best, G_best, best_loss


def fidelity(psi_a, psi_b):
    return float(np.abs(np.vdot(psi_a, psi_b)) ** 2)


# Deterministic comparison state
L_cmp, w_cmp, g_cmp = 3, 0.5, 0.5
tau_cmp, jump_idx = 0.8, 0

model_e = spin_chain_model(L=L_cmp, w=w_cmp, gamma=g_cmp)
model_g = free_fermion_gaussian_model(L=L_cmp, w=w_cmp, gamma=g_cmp)

be = ExactStateBackend(model_e, model_e.initial_state.copy())
bg = GaussianStateBackend(model_g, model_g.initial_gamma.copy())

be.propagate_no_click(tau_cmp)
bg.propagate_no_click(tau_cmp)

# Apply same jump channel if both rates are positive
rates_e = be.channel_rates()
rates_g = bg.channel_rates()
if rates_e[jump_idx] > 1e-10 and rates_g[jump_idx] > 1e-10:
    be.apply_jump(jump_idx)
    bg.apply_jump(jump_idx)

gammas = majorana_ops_jw(L_cmp)
Gamma_exact = gamma_from_psi(be.psi, gammas)
Gamma_gauss = bg.Gamma.copy()

# Covariance-level check
gamma_max_err = float(np.max(np.abs(Gamma_exact - Gamma_gauss)))

# Reconstruct wavefunction from each covariance
psi_from_Ge, Ge_fit, loss_e = reconstruct_state_from_gamma(Gamma_exact, L_cmp, n_starts=14, seed=123)
psi_from_Gg, Gg_fit, loss_g = reconstruct_state_from_gamma(Gamma_gauss, L_cmp, n_starts=14, seed=456)

fid_recon_to_exact = fidelity(psi_from_Ge, be.psi)
fid_recon_between = fidelity(psi_from_Ge, psi_from_Gg)

print('Covariance -> many-body wavefunction check')
print('-' * 72)
print(f'L={L_cmp}, w={w_cmp}, gamma={g_cmp}, tau={tau_cmp}, jump={jump_idx}')
print(f'max|Gamma_exact - Gamma_gauss| = {gamma_max_err:.3e}')
print(f'reconstruction loss from Gamma_exact = {loss_e:.3e}')
print(f'reconstruction loss from Gamma_gauss = {loss_g:.3e}')
print(f'fidelity(psi_from_Gamma_exact, exact_psi) = {fid_recon_to_exact:.8f}')
print(f'fidelity(psi_from_Gamma_exact, psi_from_Gamma_gauss) = {fid_recon_between:.8f}')
print('-' * 72)

# Optional sanity visuals
fig, ax = plt.subplots(1, 2, figsize=(10.5, 4))
ax[0].imshow(Gamma_exact, cmap='coolwarm', vmin=-1, vmax=1)
ax[0].set_title('Gamma_exact')
ax[1].imshow(Gamma_gauss, cmap='coolwarm', vmin=-1, vmax=1)
ax[1].set_title('Gamma_gauss')
plt.tight_layout()
plt.show()



## 6) Compact status line

In [ ]:
ok_backend = all(r['pass_4se'] for r in backend_rows)\nrdisc = abc_rows[2]\nprint('Current-code summary')\nprint('='*72)\nprint('Backend agreement (Exact vs Gaussian, 4*SE criterion):', ok_backend)\nprint('Representative A/B/C discrepancy at T=%.1f: TV(C,A)=%.4f, TV(C,B)=%.4f' % (rdisc['T'], rdisc['tvCA'], rdisc['tvCB']))\nprint('A vs C (w>0 Gaussian): TV=%.4f, chi2 p=%.4g' % (tv_AC, chi2_p))\nprint('EE computed with L=%d, zeta=%.2f, trajectories=%d' % (L_ee, zeta_ee, N_ee))\nprint('='*72)